## 🎯 Learning Outcomes
### By the end of this module, you will be able to

- **Array creation** and **attributes** (shape, dtype, ndim)
- **Indexing & slicing** — basic, boolean, fancy
- **Vectorized operations** and **broadcasting**
- **Universal functions** and **aggregations** (with the `axis` parameter)
- **Reshaping**, **combining**, and **splitting** arrays
- **Linear algebra** and **random sampling**
- **Persistence** and **performance considerations**

In [1]:
import numpy as np
print("NumPy version:", np.__version__)

NumPy version: 2.0.2


## Part 1 — Why NumPy?

Python lists are flexible but slow for numerical work. NumPy stores data in a **contiguous block of memory** with a single dtype and runs operations in optimized C code. The result: 10×–1000× speedups for typical numerical code.

In [2]:
import time

# -- Pure Python loop ----------------------------------------------------------
xs = list(range(1_000_000))
t0 = time.perf_counter()
ys = [x * 2 for x in xs]
t_loop = time.perf_counter() - t0

# -- NumPy vectorized ----------------------------------------------------------
arr = np.arange(1_000_000)
t0 = time.perf_counter()
arr2 = arr * 2
t_numpy = time.perf_counter() - t0

print(f"Python loop : {t_loop*1000:7.2f} ms")
print(f"NumPy       : {t_numpy*1000:7.2f} ms")
print(f"Speedup     : {t_loop/t_numpy:7.0f}x")

Python loop :   69.82 ms
NumPy       :    7.13 ms
Speedup     :      10x


## Part 2 — Array Creation

| Constructor | Returns |
|---|---|
| `np.array(seq)` | array from a Python sequence |
| `np.zeros(shape)` / `np.ones(shape)` | filled with 0 / 1 |
| `np.full(shape, v)` | filled with constant `v` |
| `np.eye(n)` | n×n identity matrix |
| `np.arange(start, stop, step)` | controls the **step** |
| `np.linspace(start, stop, n)` | controls the **count** (great for plotting) |

In [3]:
# -- From sequences ------------------------------------------------------------
a = np.array([1, 2, 3])                  # <- 1-D
b = np.array([[1, 2, 3], [4, 5, 6]])     # <- 2-D

# -- Pre-filled --------------------------------------------------------
zeros = np.zeros((3, 4))                 # 3x4 of zeros
ones  = np.ones((2, 5), dtype=np.int32)  # 2x5 of ones, int32
seven = np.full((2, 2), 7)               # filled with 7
I     = np.eye(4)                        # 4x4 identity

# -- Ranges ------------------------------------------------------------
print("arange:  ", np.arange(0, 10, 2))      # step = 2 -> [0 2 4 6 8]
print("linspace:", np.linspace(0, 1, 5))     # 5 points -> [0 .25 .5 .75 1]

# -- Like another array (same shape & dtype) ---------------------------
zl = np.zeros_like(b)                    # zeros with same shape as b
print("zeros_like b -> shape", zl.shape)

arange:   [0 2 4 6 8]
linspace: [0.   0.25 0.5  0.75 1.  ]
zeros_like b -> shape (2, 3)


## Part 3 — Array Attributes and dtypes

| Attribute | Meaning |
|---|---|
| `shape` | tuple of dimensions |
| `ndim` | number of dimensions |
| `size` | total number of elements |
| `dtype` | element type |
| `itemsize` / `nbytes` | bytes per element / total bytes |

**dtype matters** for memory and ML/GPU compatibility. `float32` halves memory vs. `float64` and is the standard on GPUs.

In [4]:
a = np.array([[1, 2, 3], [4, 5, 6]], dtype=np.float32)

print("shape    :", a.shape)         # (2, 3)
print("ndim     :", a.ndim)          # 2
print("size     :", a.size)          # 6
print("dtype    :", a.dtype)         # float32
print("itemsize :", a.itemsize, "bytes per element")
print("nbytes   :", a.nbytes, "bytes total")

# -- Memory difference between dtypes ------------------------------------------
big_f64 = np.zeros(1_000_000, dtype=np.float64)
big_f32 = np.zeros(1_000_000, dtype=np.float32)
print(f"\nfloat64 : {big_f64.nbytes/1e6:5.1f} MB")
print(f"float32 : {big_f32.nbytes/1e6:5.1f} MB   <- half the memory, GPU-friendly")

shape    : (2, 3)
ndim     : 2
size     : 6
dtype    : float32
itemsize : 4 bytes per element
nbytes   : 24 bytes total

float64 :   8.0 MB
float32 :   4.0 MB   <- half the memory, GPU-friendly


## Part 4 — Indexing and Slicing

NumPy uses **comma-separated** indexing for multi-dimensional arrays. Slicing returns a **view** (not a copy) — mutating the view mutates the original.

In [5]:
a = np.array([[ 1,  2,  3,  4],
              [ 5,  6,  7,  8],
              [ 9, 10, 11, 12]])

# -- Element / row / column / submatrix ----------------------------------------
print("a[0, 0]      =", a[0, 0])              # single element
print("a[1]         =", a[1])                 # entire row
print("a[:, 2]      =", a[:, 2])              # entire column
print("a[0:2, 1:3]  =\n", a[0:2, 1:3])       # submatrix

# -- Slices are VIEWS, not copies ----------------------------------------------
view = a[0:2, 1:3]
view[0, 0] = 999
print("\na after view mutation:")
print(a)        # <- the original was changed via the view!

# Use .copy() to detach from the original
indep = a[0:2, 1:3].copy()

a[0, 0]      = 1
a[1]         = [5 6 7 8]
a[:, 2]      = [ 3  7 11]
a[0:2, 1:3]  =
 [[2 3]
 [6 7]]

a after view mutation:
[[  1 999   3   4]
 [  5   6   7   8]
 [  9  10  11  12]]


In [6]:
# -- 3D arrays follow the same pattern -----------------------------------------
cube = np.arange(60).reshape(3, 4, 5)        # shape (depth, rows, cols)
print("cube.shape       :", cube.shape)
print("cube[0].shape    :", cube[0].shape)        # one slab
print("cube[:, 0, :].shape:", cube[:, 0, :].shape)  # top row of every slab
print("cube[..., 0].shape :", cube[..., 0].shape)   # ... = 'all earlier axes'

cube.shape       : (3, 4, 5)
cube[0].shape    : (4, 5)
cube[:, 0, :].shape: (3, 5)
cube[..., 0].shape : (3, 4)


## Part 5 — Boolean and Fancy Indexing

| Pattern | What it does |
|---|---|
| `a[mask]` (mask is a boolean array) | **filter** — keep elements where mask is True |
| `a[idx]` (idx is a list of ints) | **gather** — pick by integer position |
| `a[mask] = value` | bulk-assign matching elements |

These two patterns replace almost every Python `for`-loop in data work.

In [7]:
a = np.array([10, 20, 30, 40, 50])

# -- Boolean indexing (filter) -------------------------------------------------
mask = a > 25
print("mask         :", mask)
print("a[a > 25]    :", a[a > 25])

# -- Combine conditions with & and | (NOT 'and' / 'or') ------------------------
print("a[(a > 10) & (a < 50)] :", a[(a > 10) & (a < 50)])

# -- Fancy indexing (gather by position) ---------------------------------------
print("a[[0, 2, 4]] :", a[[0, 2, 4]])

# -- Bulk assignment via mask --------------------------------------------------
b = a.copy()
b[b > 25] = 0
print("b after mask-assign  :", b)

mask         : [False False  True  True  True]
a[a > 25]    : [30 40 50]
a[(a > 10) & (a < 50)] : [20 30 40]
a[[0, 2, 4]] : [10 30 50]
b after mask-assign  : [10 20  0  0  0]


In [8]:
# -- np.where: vectorized if/else ----------------------------------------------
scores = np.array([55, 72, 88, 91, 47, 65])
grades = np.where(scores >= 60, "pass", "fail")          # element-wise if/else
print(grades)

# Three-level mapping with np.select
conds  = [scores < 60, scores < 80, scores >= 80]
labels = ["F",         "C/B",      "A"]
print(np.select(conds, labels, default='?'))

['fail' 'pass' 'pass' 'pass' 'fail' 'pass']
['F' 'C/B' 'A' 'A' 'F' 'C/B']


## Part 6 — Element-wise Operations and Broadcasting

Arithmetic on arrays is **element-wise**. When shapes differ, NumPy uses **broadcasting** to align them.

**Broadcasting rule**: comparing dimensions **from the right**, each pair must be either equal *or* one of them must be 1. The size-1 dimension is then stretched to match.

In [9]:
a = np.array([1, 2, 3])
b = np.array([10, 20, 30])

print("a + b     :", a + b)      # element-wise
print("a * b     :", a * b)
print("a ** 2    :", a ** 2)
print("a + 100   :", a + 100)    # scalar broadcasts to every element

a + b     : [11 22 33]
a * b     : [10 40 90]
a ** 2    : [1 4 9]
a + 100   : [101 102 103]


In [10]:
# -- 2D + 1D: row-wise broadcast -----------------------------------------------
M = np.ones((3, 4))                  # shape (3, 4)
v = np.array([1, 2, 3, 4])           # shape (4,)
# Compare from right: (3, 4) vs (4,) -> (3, 4) vs (1, 4) -> compatible
print("M + v -> shape", (M + v).shape)
print(M + v)

# -- Outer product via broadcasting --------------------------------------------
col = np.array([1, 2, 3]).reshape(3, 1)        # shape (3, 1)
row = np.array([10, 20, 30, 40]).reshape(1, 4)  # shape (1, 4)
# (3, 1) and (1, 4) -> both stretch -> (3, 4)
print("\ncol * row -> shape (3, 4):")
print(col * row)

M + v -> shape (3, 4)
[[2. 3. 4. 5.]
 [2. 3. 4. 5.]
 [2. 3. 4. 5.]]

col * row -> shape (3, 4):
[[ 10  20  30  40]
 [ 20  40  60  80]
 [ 30  60  90 120]]


## Part 7 — Universal Functions and Aggregations

**Ufuncs** apply element-wise. **Aggregations** reduce an array along an axis.

| Operation | Examples |
|---|---|
| ufuncs (element-wise) | `np.sqrt`, `np.exp`, `np.log`, `np.abs`, `np.maximum` |
| reductions (whole array) | `.sum()`, `.mean()`, `.std()`, `.min()`, `.argmax()` |
| reductions (per-axis) | same methods with `axis=0` (down columns) or `axis=1` (across rows) |

In [11]:
a = np.array([[1, 2, 3], [4, 5, 6]])

# -- Ufuncs (element-wise) -----------------------------------------------------
print("sqrt        :", np.sqrt(a).round(2))
print("exp         :", np.exp(a).round(2))
print("log         :", np.log(a).round(2))
print("max(a, 3)   :", np.maximum(a, 3))         # element-wise max with 3

# -- Reductions over the entire array ------------------------------------------
print("\nsum   :", a.sum())
print("mean  :", a.mean())
print("std   :", a.std().round(3))
print("argmax:", a.argmax())     # index in the FLATTENED array

sqrt        : [[1.   1.41 1.73]
 [2.   2.24 2.45]]
exp         : [[  2.72   7.39  20.09]
 [ 54.6  148.41 403.43]]
log         : [[0.   0.69 1.1 ]
 [1.39 1.61 1.79]]
max(a, 3)   : [[3 3 3]
 [4 5 6]]

sum   : 21
mean  : 3.5
std   : 1.708
argmax: 5


In [12]:
# -- The axis parameter is the dimension that DISAPPEARS -----------------------
print("a:")
print(a)

# axis=0  -> collapse rows -> result has 1 row, ncols columns
print("\na.sum(axis=0)  -> sum DOWN columns :", a.sum(axis=0))     # shape (3,)

# axis=1  -> collapse cols -> result has nrows rows, 1 column
print("a.sum(axis=1)  -> sum ACROSS rows  :", a.sum(axis=1))        # shape (2,)

# Cumulative aggregations preserve shape
print("cumsum(axis=1) :")
print(a.cumsum(axis=1))

a:
[[1 2 3]
 [4 5 6]]

a.sum(axis=0)  -> sum DOWN columns : [5 7 9]
a.sum(axis=1)  -> sum ACROSS rows  : [ 6 15]
cumsum(axis=1) :
[[ 1  3  6]
 [ 4  9 15]]


## Part 8 — Reshaping and Combining Arrays

In [13]:
a = np.arange(12)

# -- Reshape -------------------------------------------------------------------
print("reshape(3, 4):")
print(a.reshape(3, 4))

# -1 means 'infer this dimension from the others'
print("\nreshape(3, -1):")
print(a.reshape(3, -1))

# ravel -> view (1-D), flatten -> always a copy
print("\nravel:    ", a.reshape(3, 4).ravel())

reshape(3, 4):
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]

reshape(3, -1):
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]

ravel:     [ 0  1  2  3  4  5  6  7  8  9 10 11]


In [14]:
M = np.array([[1, 2], [3, 4]])
N = np.array([[5, 6], [7, 8]])

# -- Combine -------------------------------------------------------------------
print("vstack (rows):")
print(np.vstack([M, N]))          # rows of M then rows of N

print("\nhstack (cols):")
print(np.hstack([M, N]))          # M side-by-side with N

print("\nstack (NEW axis):")
print(np.stack([M, N]).shape)     # (2, 2, 2) -> a 'depth' axis added

# -- Split ---------------------------------------------------------------------
print("\nsplit:")
parts = np.split(np.arange(12), 3)
for p in parts:
    print("  ", p)

vstack (rows):
[[1 2]
 [3 4]
 [5 6]
 [7 8]]

hstack (cols):
[[1 2 5 6]
 [3 4 7 8]]

stack (NEW axis):
(2, 2, 2)

split:
   [0 1 2 3]
   [4 5 6 7]
   [ 8  9 10 11]


## Part 9 — Linear Algebra and Random Numbers

`np.linalg` covers matrix algebra. `np.random.default_rng()` is the modern API for sampling.

In [15]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

# -- Matrix operations ---------------------------------------------------------
print("A @ B (matmul):")
print(A @ B)

print("\ninv(A):")
print(np.linalg.inv(A).round(3))

print("\ndet(A):", np.linalg.det(A).round(3))
print("||A||  :", np.linalg.norm(A).round(3))

# -- Solve a linear system  Ax = b  --------------------------------------------
b = np.array([5, 11])
x = np.linalg.solve(A, b)
print("\nsolve Ax = b -> x =", x)

A @ B (matmul):
[[19 22]
 [43 50]]

inv(A):
[[-2.   1. ]
 [ 1.5 -0.5]]

det(A): -2.0
||A||  : 5.477

solve Ax = b -> x = [1. 2.]


In [16]:
# -- Random sampling: modern API -----------------------------------------------
rng = np.random.default_rng(seed=42)             # <- seed for reproducibility

print("normal(0, 1, 5)        :", rng.normal(0, 1, 5).round(3))
print("uniform(0, 10, 5)      :", rng.uniform(0, 10, 5).round(2))
print("integers(0, 100, 5)    :", rng.integers(0, 100, 5))
print("choice(['A','B','C'], 5):", rng.choice(['A','B','C'], 5))

# -- Shuffling -----------------------------------------------------------------
arr = np.arange(10)
rng.shuffle(arr)                                 # in-place
print("shuffled:", arr)

normal(0, 1, 5)        : [ 0.305 -1.04   0.75   0.941 -1.951]
uniform(0, 10, 5)      : [9.76 7.61 7.86 1.28 4.5 ]
integers(0, 100, 5)    : [50 37 18 92 78]
choice(['A','B','C'], 5): ['B' 'B' 'C' 'B' 'B']
shuffled: [6 9 1 5 4 0 8 7 3 2]


## Part 10 — Persistence and Performance Tips

In [17]:
arr = np.arange(20).reshape(4, 5)

# -- Binary format (fast, preserves dtype/shape) -------------------------------
np.save('/tmp/arr.npy', arr)
loaded = np.load('/tmp/arr.npy')
print("Loaded back:")
print(loaded)
print("match:", np.array_equal(arr, loaded))

# -- Text format (human readable) ----------------------------------------------
np.savetxt('/tmp/arr.csv', arr, delimiter=',', fmt='%d')
print("\nFile content:")
print(open('/tmp/arr.csv').read())

Loaded back:
[[ 0  1  2  3  4]
 [ 5  6  7  8  9]
 [10 11 12 13 14]
 [15 16 17 18 19]]
match: True

File content:
0,1,2,3,4
5,6,7,8,9
10,11,12,13,14
15,16,17,18,19



In [18]:
# -- Performance: vectorize, choose dtype, avoid Python loops ------------------
N = 1_000_000
x = np.random.default_rng(0).normal(size=N).astype(np.float32)

# Vectorized expression - one C-level loop
t0 = time.perf_counter()
y = np.where(x > 0, np.sqrt(x), 0)
print(f"vectorized : {(time.perf_counter()-t0)*1000:.2f} ms")

# Equivalent Python loop - DON'T do this
t0 = time.perf_counter()
y2 = np.empty_like(x)
for i in range(N):
    y2[i] = np.sqrt(x[i]) if x[i] > 0 else 0
print(f"python loop: {(time.perf_counter()-t0)*1000:.2f} ms")

vectorized : 13.55 ms


/tmp/ipykernel_476/3254961719.py:7: RuntimeWarning: invalid value encountered in sqrt
  y = np.where(x > 0, np.sqrt(x), 0)


python loop: 952.27 ms


## Part 11 — Reading CSV Files with NumPy

Real-world data almost always starts as a **CSV file**. NumPy can load one directly with `np.genfromtxt()`, which handles column names, missing values, and mixed types. For purely numeric data you can also use the lighter `np.loadtxt()`.

| Function | Best for |
|---|---|
| `np.loadtxt(file, delimiter=',')` | Clean numeric-only CSVs |
| `np.genfromtxt(file, delimiter=',', names=True, dtype=None, encoding='utf-8')` | CSVs with headers, strings, or missing values |

We will work with **`students_scores.csv`**, a dataset of 15 students with math, science, and English exam scores.

In [19]:
import numpy as np

# -- Load the CSV with headers and automatic type detection ------------------
data = np.genfromtxt(
    'students_scores.csv',
    delimiter=',',
    names=True,          # first row becomes column names
    dtype=None,          # auto-detect each column's dtype
    encoding='utf-8'
)

# genfromtxt returns a 'structured array': each column has a name and its own dtype
print('Shape         :', data.shape)          # (15,) — one record per row
print('Column names  :', data.dtype.names)
print('Dtypes        :', data.dtype)
print()
print('First 3 rows:')
for row in data[:3]:
    print(' ', row)

Shape         : (15,)
Column names  : ('student_id', 'name', 'age', 'math_score', 'science_score', 'english_score', 'attendance_pct', 'passed')
Dtypes        : [('student_id', '<i8'), ('name', '<U17'), ('age', '<i8'), ('math_score', '<i8'), ('science_score', '<i8'), ('english_score', '<i8'), ('attendance_pct', '<f8'), ('passed', '?')]

First 3 rows:
  (1, 'Ali Hassan', 18, 88, 92, 75, 96.5, True)
  (2, 'Sara Al-Mutairi', 19, 72, 68, 84, 88.0, True)
  (3, 'Khalid Nasser', 18, 55, 60, 58, 74.5, False)


### Accessing columns and rows

In a **structured array** every column is accessed by name, just like a dictionary key. You can also slice rows with normal integer indexing.

In [20]:
# -- Access individual columns by name ------------------------------------
math_scores    = data['math_score'].astype(float)
science_scores = data['science_score'].astype(float)
english_scores = data['english_score'].astype(float)
attendance     = data['attendance_pct'].astype(float)
names          = data['name']

print('Math scores   :', math_scores)
print('Student names :', names)

# -- Slice rows: first 5 students -----------------------------------------
print('\nFirst 5 students (name, math):')
for name, score in zip(names[:5], math_scores[:5]):
    print(f'  {name:<22} {score:.0f}')

Math scores   : [88. 72. 55. 95. 43. 78. 66. 90. 38. 82. 60. 91. 47. 76. 85.]
Student names : ['Ali Hassan' 'Sara Al-Mutairi' 'Khalid Nasser' 'Fatima Al-Zahra'
 'Omar Bin Saleh' 'Noura Al-Rashidi' 'Youssef Khalil' 'Maryam Ibrahim'
 'Tariq Al-Jabri' 'Hessa Al-Mansoori' 'Bilal Farouk' 'Lina Mahmoud'
 'Saad Al-Otaibi' 'Reem Al-Hamdan' 'Ziad Al-Harbi']

First 5 students (name, math):
  Ali Hassan             88
  Sara Al-Mutairi        72
  Khalid Nasser          55
  Fatima Al-Zahra        95
  Omar Bin Saleh         43


### Descriptive statistics on CSV columns

Once you have NumPy arrays from the CSV you can apply all the aggregations you already know.

In [21]:
# -- Stack the three numeric score columns into a 2-D matrix (15 x 3) -----
scores = np.column_stack([math_scores, science_scores, english_scores])
subjects = ['Math', 'Science', 'English']

print(f'{'Subject':<12} {'Mean':>6} {'Std':>6} {'Min':>6} {'Max':>6}')
print('-' * 40)
for subj, col in zip(subjects, scores.T):          # .T iterates over columns
    print(f'{subj:<12} {col.mean():6.1f} {col.std():6.1f} {col.min():6.0f} {col.max():6.0f}')

# -- Overall average per student (mean across 3 subjects) ---------------------
overall_avg = scores.mean(axis=1)               # axis=1 -> average across columns
print('\nOverall average per student (first 5):')
for name, avg in zip(names[:5], overall_avg[:5]):
    print(f'  {name:<22} {avg:.1f}')

Subject        Mean    Std    Min    Max
----------------------------------------
Math           71.1   18.0     38     95
Science        73.2   16.0     45     97
English        73.1   15.4     44     93

Overall average per student (first 5):
  Ali Hassan             85.0
  Sara Al-Mutairi        74.7
  Khalid Nasser          57.7
  Fatima Al-Zahra        94.3
  Omar Bin Saleh         51.7


### Filtering rows with boolean indexing

Boolean masks work exactly the same on structured arrays — combine any column condition to pick the rows you want.

In [22]:
# -- Find students who scored below 60 in Math ----------------------------
low_math_mask = math_scores < 60
print('Students with Math score < 60:')
for name, score in zip(names[low_math_mask], math_scores[low_math_mask]):
    print(f'  {name:<22} scored {score:.0f}')

# -- High performers: Math AND Science both >= 85 -----------------------------
high_mask = (math_scores >= 85) & (science_scores >= 85)
print('\nHigh performers (Math >= 85 AND Science >= 85):')
for name, m, s in zip(names[high_mask], math_scores[high_mask], science_scores[high_mask]):
    print(f'  {name:<22} Math={m:.0f}  Science={s:.0f}')

# -- Attendance below 70 % — at-risk students ---------------------------------
at_risk_mask = attendance < 70
print(f'\nAt-risk students (attendance < 70 %): {at_risk_mask.sum()} found')
for name, att in zip(names[at_risk_mask], attendance[at_risk_mask]):
    print(f'  {name:<22} {att:.1f} %')

Students with Math score < 60:
  Khalid Nasser          scored 55
  Omar Bin Saleh         scored 43
  Tariq Al-Jabri         scored 38
  Saad Al-Otaibi         scored 47

High performers (Math >= 85 AND Science >= 85):
  Ali Hassan             Math=88  Science=92
  Fatima Al-Zahra        Math=95  Science=97
  Maryam Ibrahim         Math=90  Science=88
  Lina Mahmoud           Math=91  Science=95

At-risk students (attendance < 70 %): 3 found
  Omar Bin Saleh         65.0 %
  Tariq Al-Jabri         58.0 %
  Saad Al-Otaibi         61.0 %


### Computing and adding a derived column

Use vectorised arithmetic to create a new metric — a **weighted average** — without any Python loop.

In [23]:
# -- Weighted average: Math 40 %, Science 35 %, English 25 % --------------
weights    = np.array([0.40, 0.35, 0.25])
weighted   = scores @ weights          # matrix-vector multiply: (15,3) @ (3,) -> (15,)

# -- Find the top 3 students ---------------------------------------------------
top3_idx   = np.argsort(weighted)[::-1][:3]          # argsort ascending; reverse; top 3
print('Top 3 students by weighted average:')
for rank, idx in enumerate(top3_idx, start=1):
    print(f'  {rank}. {names[idx]:<22} weighted avg = {weighted[idx]:.1f}')

# -- Class-wide pass rate (passed column is stored as bool/bytes) -------------
passed     = data['passed'].astype(bool)
pass_count = passed.sum()                         # True counts as 1 in NumPy sums
print(f'\nPass rate: {pass_count}/{len(passed)} students ({pass_count/len(passed)*100:.0f} %)')

Top 3 students by weighted average:
  1. Fatima Al-Zahra        weighted avg = 94.7
  2. Lina Mahmoud           weighted avg = 91.9
  3. Maryam Ibrahim         weighted avg = 90.0

Pass rate: 11/15 students (73 %)


> **Key takeaways — CSV + NumPy**
>
> | Step | Code |
> |---|---|
> | Load with headers | `np.genfromtxt(file, delimiter=',', names=True, dtype=None, encoding='utf-8')` |
> | Extract a column | `data['column_name'].astype(float)` |
> | Stack into matrix | `np.column_stack([col1, col2, col3])` |
> | Descriptive stats | `.mean()`, `.std()`, `.min()`, `.max()` with `axis=` |
> | Filter rows | Boolean mask: `data[col > threshold]` |
> | Weighted score | `scores @ weights` (matrix-vector multiply) |
> | Top-N | `np.argsort(arr)[::-1][:N]` |
>
> For CSVs with many columns, mixed types, or heavy transformations, **pandas** is often more convenient — but the NumPy workflow above is fully sufficient for numerical analysis and integrates naturally with everything you learned in Parts 1–10.

---

## Worked Examples

The examples below combine multiple concepts from above into small, realistic mini-projects.

### Example 1 — Standardize a feature matrix

A common ML preprocessing step: subtract the column mean and divide by the column std, so every feature has mean 0 and std 1. Pure broadcasting, no loops.

In [24]:
# A 6-row, 3-column dataset where each column is on a wildly different scale
X = np.array([
    [ 1.0, 100.0,  0.01],
    [ 2.0, 200.0,  0.02],
    [ 3.0, 150.0,  0.03],
    [ 4.0, 300.0,  0.04],
    [ 5.0, 250.0,  0.05],
    [ 6.0, 400.0,  0.06],
])

mu    = X.mean(axis=0)        # per-column mean      shape (3,)
sigma = X.std(axis=0)         # per-column std       shape (3,)

# Broadcasting: (6, 3) - (3,) -> (6, 3) - (1, 3) -> compatible
X_std = (X - mu) / sigma

print("Original column means :", X.mean(axis=0).round(3))
print("Standardized means    :", X_std.mean(axis=0).round(3))   # ~ 0
print("Standardized stds     :", X_std.std(axis=0).round(3))    # ~ 1
print("\nStandardized X:")
print(X_std.round(3))

Original column means : [3.50000e+00 2.33333e+02 3.50000e-02]
Standardized means    : [-0. -0. -0.]
Standardized stds     : [1. 1. 1.]

Standardized X:
[[-1.464 -1.352 -1.464]
 [-0.878 -0.338 -0.878]
 [-0.293 -0.845 -0.293]
 [ 0.293  0.676  0.293]
 [ 0.878  0.169  0.878]
 [ 1.464  1.69   1.464]]


### Example 2 — Multiplication table via broadcasting

Build a 10×10 multiplication table without a single Python loop.

In [25]:
n = 10
rows = np.arange(1, n + 1).reshape(n, 1)     # column vector  (10, 1)
cols = np.arange(1, n + 1).reshape(1, n)     # row vector     (1, 10)
table = rows * cols                          # broadcast  ->   (10, 10)

print(table)

[[  1   2   3   4   5   6   7   8   9  10]
 [  2   4   6   8  10  12  14  16  18  20]
 [  3   6   9  12  15  18  21  24  27  30]
 [  4   8  12  16  20  24  28  32  36  40]
 [  5  10  15  20  25  30  35  40  45  50]
 [  6  12  18  24  30  36  42  48  54  60]
 [  7  14  21  28  35  42  49  56  63  70]
 [  8  16  24  32  40  48  56  64  72  80]
 [  9  18  27  36  45  54  63  72  81  90]
 [ 10  20  30  40  50  60  70  80  90 100]]


### Example 3 — Monte-Carlo estimate of pi

Sample random points in the unit square; the fraction inside the unit circle is pi/4.

In [26]:
rng = np.random.default_rng(2026)
N = 200_000

# -- Sample N points uniformly in the unit square ------------------------------
xy = rng.uniform(-1, 1, size=(N, 2))           # shape (N, 2)

# -- Compute squared distance from origin (vectorized) -------------------------
r2 = (xy ** 2).sum(axis=1)                     # shape (N,)

# -- Fraction inside the unit circle ------------------------------------------
inside = r2 <= 1                               # boolean mask
pi_est = 4 * inside.mean()                     # mean of bools = fraction True

print(f"N         = {N:,}")
print(f"pi (true) = {np.pi:.6f}")
print(f"pi (est)  = {pi_est:.6f}")
print(f"error     = {abs(pi_est - np.pi):.6f}")

N         = 200,000
pi (true) = 3.141593
pi (est)  = 3.143620
error     = 0.002027


### Example 4 — Solve least-squares via the normal equations

Fit a line `y = ax + b` to noisy data using only NumPy linear algebra.

In [27]:
rng = np.random.default_rng(0)

# -- Synthesise data: y = 2.5 x + 1 + noise ------------------------------------
x = np.linspace(0, 10, 50)
y = 2.5 * x + 1 + rng.normal(0, 1, size=50)

# -- Build the design matrix [x, 1] for parameters [a, b] ----------------------
A = np.column_stack([x, np.ones_like(x)])      # shape (50, 2)

# -- Solve A @ [a, b] = y in least-squares sense -------------------------------
# np.linalg.lstsq is the recommended way; here's also the manual normal eqn
params, *_ = np.linalg.lstsq(A, y, rcond=None)
a_hat, b_hat = params

print(f"True line     : y = 2.500 x + 1.000")
print(f"Recovered line: y = {a_hat:.3f} x + {b_hat:.3f}")

True line     : y = 2.500 x + 1.000
Recovered line: y = 2.616 x + 0.549


---
# Summary

| Concept | Key Points |
|---|---|
| **Why NumPy?** | Provides significant speedups over Python lists for numerical operations by storing data contiguously and using optimized C code. |
| **Array Creation** | Use `np.array()` for Python sequences; `np.zeros()`, `np.ones()`, `np.full()`, `np.eye()` for pre-filled arrays; `np.arange()` (specifies step) and `np.linspace()` (specifies count) for ranges; `_like` functions for replicating shape/dtype. |
| **Attributes & dtypes** | Essential attributes include `shape`, `ndim`, `size`, `dtype`, `itemsize`, and `nbytes`. `dtype` choice (e.g., `float32` vs. `float64`) impacts memory efficiency and compatibility. |
| **Indexing & Slicing** | Uses comma-separated indices for multi-dimensional arrays. Slices return a **view** of the original array; use `.copy()` to create an independent copy. |
| **Boolean Indexing** | Allows **filtering** elements based on a boolean mask (e.g., `a[a > 25]`). Conditions can be combined using `&` (AND) and `|` (OR). |
| **Fancy Indexing** | Enables **gathering** elements by providing a list of integer positions (e.g., `a[[0, 2, 4]]`). Useful for non-contiguous selection and bulk assignment. |
| **`np.where` & `np.select`** | Provide vectorized ways to apply conditional logic element-wise, similar to if/else statements. |
| **Element-wise Operations** | Basic arithmetic operations (`+`, `-`, `*`, `/`, `**`) are applied to arrays element by element. |
| **Broadcasting** | A mechanism where NumPy handles operations on arrays with different shapes. It aligns dimensions from the right; dimensions must be equal or one must be 1. |
| **Universal Functions (Ufuncs)** | Functions that operate element-wise on arrays (e.g., `np.sqrt()`, `np.exp()`, `np.log()`, `np.maximum()`). |
| **Aggregations / Reductions** | Functions that reduce an array to a single value (e.g., `.sum()`, `.mean()`, `.std()`, `.min()`, `.argmax()`). The `axis` parameter specifies the dimension along which to aggregate. `cumsum()` performs cumulative aggregation. |
| **Reshaping** | Change the dimensions of an array using `.reshape()`. `-1` can be used to infer a dimension. `ravel()` provides a 1-D view, while `flatten()` returns a 1-D copy. `.T` transposes an array. |
| **Combining Arrays** | Arrays can be stacked vertically (`np.vstack()`), horizontally (`np.hstack()`), or along a new axis (`np.stack()`). `np.concatenate()` offers general concatenation. |
| **Splitting Arrays** | `np.split()` divides an array into multiple sub-arrays along a specified axis. |
| **Linear Algebra** | The `np.linalg` module provides functions for matrix multiplication (`@`), inverse (`inv`), determinant (`det`), norm (`norm`), and solving linear systems (`solve`). |
| **Random Sampling** | Use `np.random.default_rng(seed)` for modern, reproducible random number generation. Methods include `normal()`, `uniform()`, `integers()`, `choice()`, and `shuffle()`. |
| **Persistence** | Save arrays efficiently in binary format (`.npy`) using `np.save()` and `np.load()`. Text format (`.csv`) can be used with `np.savetxt()` and `np.loadtxt()` for human readability. |
| **Performance Tips** | Prioritize vectorized operations, choose appropriate `dtype` (e.g., `float32`), and avoid explicit Python loops for better performance.

### Contributed by: Abdulhadi Zubailah